In [1]:
from sklearn.neighbors import NearestNeighbors
import glob
import pandas as pd
from astropy.cosmology import Planck18
from astropy import units as u
import numpy as np
from astropy.coordinates import SkyCoord
from scipy.stats import norm
import matplotlib.pyplot as plt

In [25]:
target = pd.read_csv('../all_sample_20260702.csv')
tg_d = Planck18.comoving_distance(target.paliya_Z).value
tg_ra = (target.paliya_RA).values
tg_dec = (target.paliya_DEC).values

coord = SkyCoord(ra = tg_ra * u.deg, dec = tg_dec * u.deg, distance = tg_d * u.Mpc)
x = coord.cartesian.x.value
y = coord.cartesian.y.value
z = coord.cartesian.z.value
tg_df = pd.DataFrame({'x':x, 'y':y, 'z':z})

In [ ]:
index = np.zeros([len(tg_df)])
for i in range(len(tg_df)):
    file_list = glob.glob(f'../data/catalog/ps1/redshift_STRM/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
    df = pd.read_csv(file_list[0])
    surround_z = (df[f'z_phot']).values
    surround_err = (df[f'z_photErr']).values
    surround_scale = Planck18.comoving_distance(surround_z + surround_err/2) - Planck18.comoving_distance(surround_z - surround_err/2)
    surround_d = Planck18.comoving_distance(surround_z).value

    surround_ra = (df.raMean).values
    surround_ra = surround_ra[np.isnan(surround_d) == False]
    surround_dec = (df.decMean).values
    surround_dec = surround_dec[np.isnan(surround_d) == False]
    surround_d = surround_d[np.isnan(surround_d) == False]
    surround_odds = np.abs(norm.cdf(tg_d[i] - (5*u.Mpc).value, loc =surround_d, scale = surround_scale) - norm.cdf(tg_d[i] + (5*u.Mpc).value, loc = surround_d, scale = surround_scale))

    coord = SkyCoord(ra =surround_ra * u.deg, dec = surround_dec * u.deg, distance = surround_d * u.Mpc)
    x = coord.cartesian.x.value
    y = coord.cartesian.y.value
    z = coord.cartesian.z.value

    df = pd.DataFrame({'x':x, 'y':y, 'z':z})
    nn = NearestNeighbors(n_neighbors= len(df), algorithm= 'auto')
    nn.fit(df) 
    neighbors = nn.kneighbors(tg_df.loc[[i]])
    k_index = np.cumsum([surround_odds[i] for i in neighbors[1]])
    k_index = np.where(k_index >= 5)[0]
    try:
        index[i] = neighbors[1][0][k_index[0]]
    except:
        print(f'{tg_ra[i]}, {tg_dec[i]} don\'t have enough data')
        index[i] = -999

TypeError: Cannot convert 'complex' with non-zero imaginary component to 'double' (this most likely comes from the '**' operator; use 'cython.cpow(True)' to return 'nan' instead of a complex number).

In [52]:
llr_arr = index_df.index_llr[0].astype('int')
xgb_arr = index_df.index_xgb[0].astype('int')

In [87]:
for i, idx in enumerate(xgb_arr):
    if idx == -999:
        continue
    file_list = glob.glob(f'../data/catalog/ps1/merged/t{tg_ra[i]:08.4f}{tg_dec[i]:+07.4f}.csv')
    df = pd.read_csv(file_list[0])
    df = df[['D_L_Mpc_xgb_origin', 'raMean', 'decMean']].loc[idx]
    coords = SkyCoord(ra = df.raMean * u.deg, dec = df.decMean * u.deg, distance = df.D_L_Mpc_xgb_origin * u.Mpc)
    coords = np.array([coords.cartesian.x.value, coords.cartesian.y.value, coords.cartesian.z.value])

    fifth_dis = np.linalg.norm(tg_df.loc[i].values - coords)
    fifth_dens =  5 / np.pi /fifth_dis**2
    fifth_dens = np.log(fifth_dens)
    print(i, fifth_dens)

0 -9.891372749105358
1 -8.151536622000895
2 -7.964547488891854
3 -11.083432644930722
4 -8.394449406746899
5 -9.285322172067305
6 -8.969165872079898
7 -8.95176936065619
8 -9.12933040566606
9 -8.92381910161809
10 -9.050362689768612
11 -8.985155987837508
12 -9.279619741803156
13 -10.039200938800736
14 -8.734905869119107
15 -10.198096790870034
16 -9.82851470009465
17 -8.299002349824582
18 -7.6106461100166625
19 -9.476033352504192
20 -10.190841628243268
21 -8.252714478592079
22 -9.659783564952559
23 -8.048914014402394
24 -8.609707496846143
25 -8.02891439327684
26 -9.381997443903067
27 -8.613252616756649
28 -8.062280543786292
29 -9.363605328762736
30 -8.828028921656838
31 -8.630800147650124
32 -8.1765131504312
33 -8.647704505372838
34 -8.199400935037774
35 -5.673158641483604
36 -9.114745192020518
37 -9.207473350060516
38 -7.776383773148577
39 -7.214622243451552
40 -7.867385854690539
41 -9.101453201053237
42 -8.98595591748139
43 -8.535894056705011
44 -9.596492939748972
45 -8.340167065659534
4